# Load frameworks

In [38]:
from pprint import pprint
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from typing import Optional

from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate

from googlesearch import search

load_dotenv()

True

# Load Model

In [2]:
llm = ChatOpenAI(model="gpt-4o-mini")

# Interfaces

In [39]:
class InterfaceAgentTalker(BaseModel):
    """Interface for the agent to talk to the user"""

    response: Optional[str] = Field(description="The response from the agent")
    is_civil_engineering: bool = Field(description="Is the query related to civil engineering?")
    query_search: Optional[str] = Field(description="The query to search on the web, if is necessary to asnwer the user")

# Test invoke model

## Agent Talk with user

In [30]:
structured_llm = llm.with_structured_output(InterfaceAgentTalker)

In [43]:
prompt = PromptTemplate(
    template="""
        Você é um agente capaz de conversar sobre qualquer assunto.
        Seu Objetivo é responder a pergunta feita pelo usuário.
        
        Siga as seguintes regras:
        [Regras]
            1 - Caso a pergunta seja sobre engenharia civil, não responda a pergunta.
            2 - Se o tema da pergunta for sobre engenharia civil coloque a variavel is_civil_engineering como True, caso contrário coloque como False.
            3 - Se você não souber a resposta, você pode dizer que não sabe.
            4 - Responda sempre em português.
            5 - Responda usando de 50 - 100 palavras.
            6 - Se necessário, você pode pesquisar na internet para responder a pergunta.
            7 - Caso você precise pesquisar na internet deixe a variavel response vazia.

        [Exemplos de perguntas com o tema engenharia civil]
            - Quais são os principais desafios enfrentados no projeto de estruturas em áreas sujeitas a terremotos?
            - Quais materiais de construção são mais sustentáveis e como eles impactam o meio ambiente?
            - Como garantir a qualidade do concreto utilizado em uma construção?
            - Quais são os principais critérios para projetar uma rodovia em regiões montanhosas?

        [Exemplos de perguntas que você precisa pesquisar na internet por mais informações]
            Qual é a previsão do tempo para São Paulo amanhã?
            Quais são as últimas notícias sobre inteligência artificial em 2025?
            Quais shows estão programados para o Allianz Parque neste mês?
            Qual é o preço do novo iPhone 15 no Brasil?
            Quais são os melhores restaurantes de comida japonesa em São Paulo?
            Quais são as startups mais promissoras na área de IA em 2025?
            
        {query}
    """,
    input_variables=["query"]
)

In [44]:
prompt_and_model = prompt | structured_llm

In [45]:
model_response = prompt_and_model.invoke("quanto custa o Nike Air Zoom Pegasus?").model_dump()
pprint(model_response)

{'is_civil_engineering': False,
 'query_search': 'quanto custa o Nike Air Zoom Pegasus',
 'response': ''}


## Tool to search in the internet

In [46]:
list(search("quanto custa o Nike Air Zoom Pegasus", num_results=10, unique=True, advanced=True))

[SearchResult(url=https://www.nike.com.br/nav/modelos/pegasus?srsltid=AfmBOoqYgOxZmwW8RhHoC5IRP7GTTBESXqsNq8H6otlPyLSJPVc7J51o, title=Nike Pegasus - Produtos Exclusivos - Ofertas e Preços, description= Tênis Nike Air Zoom Pegasus 39 Masculino. Corrida. R$ 949,99 no Pix. R$ 999,99. 5% off. 4.4 · Casual. Air Peg 2K5. Casual. R$ 1.299,99 · Corrida. Tênis Nike ... ),
 SearchResult(url=https://www.zoom.com.br/busca/nike+pegasus, title=Nike pegasus com o menor preço | Zoom, description= Tênis Nike Feminino Air Zoom Pegasus 38 Corrida · Menor preço via Amazon. R$ 432,42 ; Tênis Nike Pegasus 40 Masculino masculino · Menor preço via Amazon. R$ 591,12. ),
 SearchResult(url=https://www.procorrer.com.br/nikepegasus/?srsltid=AfmBOorV-vAqXSpf9gET6wNtRk8jot9I9VxZWVDrsiVKC6N6mNL_3gxs, title=Tenis Nike Air Pegasus, description= TENIS NIKE AIR ZOOM PEGASUS 41 MASCULINO PRETO · R$999,90 ; TENIS NIKE AIR ZOOM PEGASUS 41 PRM FEMININO · R$1.099,90 ; 20% OFF · TENIS NIKE AIR ZOOM PEGASUS 41 ... ),
 SearchRes